# 11 — cdfmm backends versus FMM3D

This notebook compares cdfmm CUDA-full, CUDA-partial, and oneMKL CPU-static with FMM3D 2.1.0 for source-point dipole fields. Every particle is both a source and its corresponding target, and all implementations exclude the singular self interaction. FMM3D returns $\nabla\phi$, so its result is converted with $H=-\nabla\phi$.

The cdfmm plans are deliberately constructed and released one at a time. CUDA-full and CUDA-partial are never simultaneously resident on the device.

Build the required CUDA/oneMKL Python extension and install the pinned comparison dependency from the repository root before running:

```console
conda env update -n cdfmm -f environment.yml
conda env update -n cdfmm -f environment-cuda.yml
conda env update -n cdfmm -f environment-fmm3d.yml
conda activate cdfmm
cmake --fresh --preset notebooks
cmake --build --preset notebooks -j
ctest --preset notebooks
./examples/notebooks/install_fmm3d.sh
```

In [ ]:
# All user-editable parameters are collected here.
import os

N_PARTICLES = 10_000
EXPANSION_ORDER = 6
TREE_DEPTH = 2
FMM3D_EPS = 1.0e-3
REPEAT_COUNTS = [10, 100, 1000]
ACCURACY_TARGETS = 128
VECTOR_BATCH_SIZE = 25
RANDOM_SEED = 314159
CPU_THREADS = 8

# OpenMP reads this before FMM3D and oneMKL are imported.
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)


In [ ]:
import gc
import importlib.metadata
import platform
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np

import cdfmm
try:
    import fmm3dpy
except ImportError as error:
    raise RuntimeError(
        "FMM3D is missing. Run ./examples/notebooks/install_fmm3d.sh "
        "from the repository root, then restart this kernel."
    ) from error

try:
    from fmm3d_comparison import (
        fmm3d_laplace_nterms, fmm3d_source_fields, relative_error_metrics, timed_call,
        validate_source_point_geometry,
    )
except ModuleNotFoundError:
    from examples.notebooks.fmm3d_comparison import (
        fmm3d_laplace_nterms, fmm3d_source_fields, relative_error_metrics, timed_call,
        validate_source_point_geometry,
    )

missing = []
if not cdfmm.cuda_full_available():
    missing.append("CUDA_FULL (rebuild with CUDA full support)")
if not cdfmm.cuda_m2l_p2p_available():
    missing.append("CUDA_PARTIAL (rebuild with CUDA support)")
if not cdfmm.one_mkl_available():
    missing.append("oneMKL CPU-static (reconfigure with oneMKL)")
if missing:
    raise RuntimeError(
        "Required comparison backends are unavailable: " + "; ".join(missing) + ". "
        "From the repository root, run `cmake --fresh --preset notebooks` "
        "and `cmake --build --preset notebooks -j`, then restart this kernel."
    )

FMM3D_SOURCE_VERSION = "2.1.0"
fmm3d_distribution_version = importlib.metadata.version("fmm3dpy")
configuration = {
    "particles": N_PARTICLES,
    "cdfmm order/depth": f"{EXPANSION_ORDER}/{TREE_DEPTH}",
    "cdfmm backends": "CUDA_FULL, CUDA_PARTIAL, CPU_STATIC+oneMKL",
    "FMM3D source/distribution/eps/nterms": (
        f"{FMM3D_SOURCE_VERSION}/{fmm3d_distribution_version}/"
        f"{FMM3D_EPS:g}/{fmm3d_laplace_nterms(FMM3D_EPS)}"
    ),
    "CPU threads": CPU_THREADS,
    "CUDA device": cdfmm.cuda_device_description(),
    "host": platform.node(),
}
configuration

## Shared source-point geometry

There is one position array. `target_positions` is the same Python object, and the explicit identity map is checked before any plan is built. FMM3D uses `pg=2`, its source-evaluation mode, which omits each source's singular self term.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
source_positions = rng.uniform(-0.95, 0.95, size=(N_PARTICLES, 3))
target_positions = source_positions
base_moments = rng.normal(size=(N_PARTICLES, 3))
base_moments /= np.linalg.norm(base_moments, axis=1, keepdims=True)
target_source_indices = np.arange(N_PARTICLES, dtype=np.int64)
validate_source_point_geometry(
    source_positions, target_positions, target_source_indices
)
assert target_positions is source_positions
fmm3d_sources = np.asfortranarray(source_positions.T)
assert fmm3d_sources.shape == (3, N_PARTICLES)

BACKENDS = [
    ("cdfmm CUDA-full", cdfmm.ExecutionBackend.CUDA_FULL, None),
    ("cdfmm CUDA-partial", cdfmm.ExecutionBackend.CUDA_PARTIAL, None),
    ("cdfmm oneMKL CPU-static", cdfmm.ExecutionBackend.CPU_STATIC,
     cdfmm.StaticMatrixBackend.ONE_MKL),
]

def make_cdfmm_plan(backend, matrix_backend=None):
    options = cdfmm.UniformFmmOptions()
    options.expansion_order = EXPANSION_ORDER
    options.tree.max_level = TREE_DEPTH
    options.tree.root_centre = cdfmm.Vec3(0.0, 0.0, 0.0)
    options.tree.root_half_width = 1.0
    options.backend = backend
    if matrix_backend is not None:
        options.static_matrix_backend = matrix_backend
    return cdfmm.UniformFmm(source_positions, target_positions, options)

def evaluate_cdfmm(plan, moments):
    return plan.evaluate(
        moments, output="field",
        target_source_indices=target_source_indices,
    )["H"]

def release_plan(plan):
    return None

def evaluate_fmm3d(moments):
    output = fmm3dpy.lfmm3d(
        eps=FMM3D_EPS, sources=fmm3d_sources,
        dipvec=np.asfortranarray(moments.T), pg=2,
    )
    fields = fmm3d_source_fields(output)
    if fields.shape != (N_PARTICLES, 3):
        raise RuntimeError(f"FMM3D source evaluation returned {fields.shape}")
    return fields

def exact_sample_fields(moments, sample_indices):
    fields = np.empty((len(sample_indices), 3))
    for row, particle in enumerate(sample_indices):
        fields[row] = cdfmm.p2p_dipole_sum(
            source_positions[particle], source_positions, moments,
            self_index=int(particle),
        )["H"]
    return fields


## Accuracy

Each cdfmm plan is evaluated and released before constructing the next. The exact reference evaluates 128 deterministic particle identities against all sources.

In [ ]:
all_fields = {}
for label, backend, matrix_backend in BACKENDS:
    print(f"Evaluating {label} for accuracy...", flush=True)
    plan = make_cdfmm_plan(backend, matrix_backend)
    all_fields[label] = evaluate_cdfmm(plan, base_moments)
    plan = release_plan(plan)
    gc.collect()
all_fields["FMM3D"] = evaluate_fmm3d(base_moments)

sample_indices = np.linspace(
    0, N_PARTICLES - 1, min(ACCURACY_TARGETS, N_PARTICLES), dtype=int
)
exact_fields = exact_sample_fields(base_moments, sample_indices)
accuracy_rows = []
for label, fields in all_fields.items():
    metrics = relative_error_metrics(fields[sample_indices], exact_fields)
    cross = relative_error_metrics(fields, all_fields["FMM3D"])
    accuracy_rows.append((label, metrics.rms, metrics.maximum, cross.rms))

print(f"{'Implementation':30s} {'RMS vs exact':>14s} {'Max vs exact':>14s} {'RMS vs FMM3D':>14s}")
for label, rms, maximum, cross_rms in accuracy_rows:
    print(f"{label:30s} {rms:14.6e} {maximum:14.6e} {cross_rms:14.6e}")

x = np.arange(len(accuracy_rows))
width = 0.38
plt.figure(figsize=(10, 4.5))
plt.bar(x - width / 2, [max(row[1], 1e-18) for row in accuracy_rows], width, label="RMS")
plt.bar(x + width / 2, [max(row[2], 1e-18) for row in accuracy_rows], width, label="Maximum")
plt.xticks(x, [row[0] for row in accuracy_rows], rotation=15, ha="right")
plt.yscale("log")
plt.ylabel("Relative field error")
plt.title(f"Source-point accuracy at N={N_PARTICLES:,}")
plt.legend()
plt.tight_layout()
plt.show()


## One-run comparison

Warm-ups initialise CUDA, oneMKL, OpenMP, and FMM3D before timing. Construction, reused-plan evaluation, and an independently timed construction-plus-evaluation are reported for every cdfmm backend.

In [ ]:
one_run_rows = []
for label, backend, matrix_backend in BACKENDS:
    warm_plan = make_cdfmm_plan(backend, matrix_backend)
    _ = evaluate_cdfmm(warm_plan, base_moments)
    warm_plan = release_plan(warm_plan)
    gc.collect()

    def complete_run():
        one_run_plan = make_cdfmm_plan(backend, matrix_backend)
        return evaluate_cdfmm(one_run_plan, base_moments)

    _, complete_seconds = timed_call(complete_run)
    plan, setup_seconds = timed_call(
        lambda: make_cdfmm_plan(backend, matrix_backend)
    )
    _, evaluation_seconds = timed_call(
        lambda: evaluate_cdfmm(plan, base_moments)
    )
    plan = release_plan(plan)
    gc.collect()
    one_run_rows.extend([
        (label, "construction", setup_seconds),
        (label, "reused evaluation", evaluation_seconds),
        (label, "construction + evaluation", complete_seconds),
    ])

_ = evaluate_fmm3d(base_moments)
warm_batch = np.stack([base_moments, -base_moments])
_ = fmm3dpy.lfmm3d(
    eps=FMM3D_EPS, sources=fmm3d_sources,
    dipvec=np.asfortranarray(np.transpose(warm_batch, (0, 2, 1))),
    pg=2, nd=2,
)
del warm_batch
_, fmm3d_seconds = timed_call(lambda: evaluate_fmm3d(base_moments))
one_run_rows.append(("FMM3D", "complete call", fmm3d_seconds))

print(f"{'Implementation':30s} {'Measurement':28s} {'Seconds':>12s}")
for label, measurement, seconds in one_run_rows:
    print(f"{label:30s} {measurement:28s} {seconds:12.6f}")

complete_rows = [row for row in one_run_rows if row[1] in {"construction + evaluation", "complete call"}]
plt.figure(figsize=(9, 4.5))
plt.barh([row[0] for row in complete_rows], [row[2] for row in complete_rows])
plt.xscale("log")
plt.xlabel("Wall time (s)")
plt.title(f"Complete one-run comparison at N={N_PARTICLES:,}")
plt.tight_layout()
plt.show()


## Changing-moment throughput

Moment generation is outside all timings. Each cdfmm method reuses one persistent plan during its timed loop, then releases it before the next backend is constructed. FMM3D repeated calls rebuild internal state; vectorized batches share geometry within bounded batches and are not a sequential-update API.

In [ ]:
def make_moment_states(count):
    state_rng = np.random.default_rng(RANDOM_SEED + 1)
    return state_rng.normal(size=(count, N_PARTICLES, 3))

def time_cdfmm_states(states, backend, matrix_backend):
    plan = make_cdfmm_plan(backend, matrix_backend)
    _ = evaluate_cdfmm(plan, states[0])
    checksum = 0.0
    start = perf_counter()
    for moments in states:
        checksum += float(evaluate_cdfmm(plan, moments)[0, 0])
    seconds = perf_counter() - start
    plan = release_plan(plan)
    gc.collect()
    return seconds, checksum

def time_fmm3d_calls(states):
    checksum = 0.0
    start = perf_counter()
    for moments in states:
        checksum += float(evaluate_fmm3d(moments)[0, 0])
    return perf_counter() - start, checksum

def time_fmm3d_vectorized(states):
    checksum = 0.0
    start = perf_counter()
    for begin in range(0, len(states), VECTOR_BATCH_SIZE):
        batch = states[begin:begin + VECTOR_BATCH_SIZE]
        output = fmm3dpy.lfmm3d(
            eps=FMM3D_EPS, sources=fmm3d_sources,
            dipvec=np.asfortranarray(np.transpose(batch, (0, 2, 1))),
            pg=2, nd=len(batch),
        )
        fields = fmm3d_source_fields(output)
        checksum += float(np.sum(fields[:, 0, 0]))
    return perf_counter() - start, checksum

performance_rows = []
for count in REPEAT_COUNTS:
    print(f"Benchmarking {count} changing moment states...", flush=True)
    states = make_moment_states(count)
    for label, backend, matrix_backend in BACKENDS:
        seconds, checksum = time_cdfmm_states(states, backend, matrix_backend)
        performance_rows.append((label, count, seconds, checksum))
    for label, timer in [
        ("FMM3D repeated calls", time_fmm3d_calls),
        ("FMM3D vectorized batches", time_fmm3d_vectorized),
    ]:
        seconds, checksum = timer(states)
        performance_rows.append((label, count, seconds, checksum))
    del states
    gc.collect()

for method, count, seconds, checksum in performance_rows:
    if not np.isfinite(checksum):
        raise RuntimeError(f"{method} produced a non-finite checksum")
print(f"{'Method':30s} {'States':>8s} {'Total (s)':>12s} {'s/state':>12s} {'states/s':>12s}")
for method, count, seconds, _ in performance_rows:
    print(f"{method:30s} {count:8d} {seconds:12.6f} {seconds/count:12.6e} {count/seconds:12.3f}")


In [ ]:
methods = list(dict.fromkeys(row[0] for row in performance_rows))
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for method in methods:
    selected = [row for row in performance_rows if row[0] == method]
    counts = np.array([row[1] for row in selected])
    totals = np.array([row[2] for row in selected])
    axes[0].plot(counts, totals, marker="o", label=method)
    axes[1].plot(counts, totals / counts, marker="o", label=method)
    axes[2].plot(counts, counts / totals, marker="o", label=method)
for axis in axes:
    axis.set_xscale("log")
    axis.set_yscale("log")
    axis.set_xlabel("Changing moment states")
    axis.grid(True, which="both", alpha=0.25)
axes[0].set_ylabel("Total wall time (s)")
axes[1].set_ylabel("Mean seconds per state")
axes[2].set_ylabel("States per second")
axes[0].legend(fontsize=8)
fig.suptitle(f"Source-point changing-moment performance at N={N_PARTICLES:,}")
fig.tight_layout()
plt.show()

Use complete one-run timings when setup cannot be amortised. Use the three persistent cdfmm curves and ordinary FMM3D calls for sequential simulation steps. FMM3D vectorized batches measure a different, throughput-oriented workload. All cdfmm results use the same order, depth, particles, identities, and moments; the fixed order/depth and FMM3D tolerance are practical settings rather than accuracy-matched tuning.